In [2]:
# you may need to run these on fresh dev container 
# uv sync #install deps
# uv run python -m ipykernel install --user --name capstone


In [11]:
import polars as pl
import os

In [ ]:
# load data with polars from ../data/source_data/initial_dataset/*.parquet
df00 = pl.read_parquet("../../data/source_data/initial_dataset/000000000000.parquet")

SnapshotAt,package_name,package_version,package_published_at,dep_name,dep_version,MinimumDepth,github_repo
datetime[μs],str,str,datetime[μs],str,str,i64,str
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""certifi""","""2022.12.7""",3,"""jdelic/12factor-vault"""
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""charset-normalizer""","""3.1.0""",3,"""jdelic/12factor-vault"""
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""django-dbconn-retry""","""0.1.7""",1,"""jdelic/12factor-vault"""
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""hvac""","""1.1.0""",1,"""jdelic/12factor-vault"""
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""idna""","""3.4.0""",3,"""jdelic/12factor-vault"""


In [22]:


def extract_unique_packages(input_dir: str, output_path: str) -> pl.DataFrame:
    """Iterate through all parquet files to get unique package_name and github repo with min/max snapshot dates."""
    frames = []
    parquet_files = [f for f in os.listdir(input_dir) if f.endswith(".parquet")]
    for parquet_file in parquet_files:
        df = pl.read_parquet(f"{input_dir}/{parquet_file}")
        agg = df.group_by(["package_name", "github_repo"]).agg(
            pl.col("SnapshotAt").min().alias("min_snapshot_date"),
            pl.col("SnapshotAt").max().alias("max_snapshot_date"),
        )
        frames.append(agg)
    unique_packages = pl.concat(frames).group_by(["package_name", "github_repo"]).agg(
        pl.col("min_snapshot_date").min(),
        pl.col("max_snapshot_date").max(),
    )
    unique_packages = unique_packages.with_columns(
        ("github.com/" + pl.col("github_repo")).alias("github_repo")
    )
    unique_packages.write_parquet(output_path)
    return unique_packages

unique_packages = extract_unique_packages(
    input_dir="../../data/source_data/initial_dataset",
    output_path="../../data/augmented_data/unique_packages.parquet",
)

In [23]:
unique_packages.head()


package_name,github_repo,min_snapshot_date,max_snapshot_date
str,str,datetime[μs],datetime[μs]
"""ansible-lint""","""github.com/ansible/ansible-lin…",2023-04-10 21:01:49.219309,2026-02-23 21:00:56.375217
"""disk-objectstore""","""github.com/aiidateam/disk-obje…",2023-04-10 21:01:49.219309,2026-02-23 21:00:56.375217
"""strto""","""github.com/zigai/strto""",2023-05-29 21:01:38.282067,2026-02-23 21:00:56.375217
"""pypuppetdb""","""github.com/voxpupuli/pypuppetd…",2023-04-10 21:01:49.219309,2026-02-23 21:00:56.375217
"""even-glasses""","""github.com/emingenc/even_glass…",2025-02-25 04:59:51.105884,2026-02-23 21:00:56.375217


In [24]:
# get min minshapshot date and max maxsnapshot date
min_date = unique_packages.select(pl.col("min_snapshot_date").min()).item()
max_date = unique_packages.select(pl.col("max_snapshot_date").max()).item()
print(f"Min snapshot date: {min_date}")
print(f"Max snapshot date: {max_date}")

Min snapshot date: 2023-04-10 21:01:49.219309
Max snapshot date: 2026-02-23 21:00:56.375217
